# 07_generate_quality_scores_type_confidence

This notebook only performs Step 7: generate data quality base metrics, city quality scores, quality weights, local quality flags, and the `quality_confidence_component` needed for downstream `type_confidence`. This step does not run PCA, UMAP, clustering, morphotype assignment, scale stability, accessibility models, or final `type_confidence` generation.


## Part 0: Environment and paths


In [1]:
# Part 0: Environment and paths
# Note: this notebook runs with system python3, and all outputs are limited to data/07_generate_quality_scores_type_confidence/.
from pathlib import Path
from datetime import datetime
import json
import math
import platform
import sys
import warnings

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore", category=RuntimeWarning)
pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 180)

ROOT = Path("/Volumes/ZHITAI2T/202606osm")
CODE_DIR = ROOT / "code_upload/07_generate_quality_scores_type_confidence"
OUT_DIR = ROOT / "data/07_generate_quality_scores_type_confidence"
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Try to enable parquet output; if pyarrow is unavailable, this step still writes complete CSV outputs and records the fallback in the log.
try:
    import pyarrow  # noqa: F401
    PARQUET_AVAILABLE = True
    PARQUET_ENGINE = "pyarrow"
except Exception as exc:  # pragma: no cover - environment compatibility only
    PARQUET_AVAILABLE = False
    PARQUET_ENGINE = None
    print(f"pyarrow is unavailable; writing CSV only: {exc!r}")

STARTED_AT = datetime.now().isoformat(timespec="seconds")
SCORE_SCALE = "0-100"
SCORE_WEIGHTS = {
    "network_integrity_score": 0.30,
    "historical_maturity_score": 0.25,
    "poi_completeness_score": 0.15,
    "population_built_support_score": 0.10,
    "local_coverage_score": 0.20,
}
TIER_RULES = {
    "core": "quality_score >= 70, drive/walk full_city metrics are available, and no severe input gaps are present; move to sensitivity if local coverage, history, or POI support is clearly weak.",
    "sensitivity": "50 <= quality_score < 70, or local-scale valid_ratio is low, or selected POI/history/population support metrics are weak, or reviewable but explainable issues are present.",
    "excluded_candidate": "quality_score < 50, or any core full_city drive/walk network is unavailable, or key inputs are missing, or anomaly records indicate systematic unexplained issues. This label does not remove cities; it is only for review and sensitivity analysis.",
}
LOCAL_VALID_AREA_MIN = {
    "hex_1km": 0.10,
    "hex_2km": 0.05,
}

INPUT_PATHS = {
    "city_master": ROOT / "data/01_city_boundaries_sample_list/city_sample/city_master.csv",
    "population_built": ROOT / "data/03_download_population_built_environment_data/city_population_built_environment_metrics.csv",
    "unit_quality_check": ROOT / "data/05_build_multiscale_spatial_units/spatial_units_quality_checks.csv",
    "unit_index": ROOT / "data/05_build_multiscale_spatial_units/spatial_units_index.csv",
    "city_metrics": ROOT / "data/06_clean_road_networks_calculate_morphology_metrics/city_morphology_metrics.csv",
    "local_metrics": ROOT / "data/06_clean_road_networks_calculate_morphology_metrics/local_morphology_metrics.csv",
    "metrics_quality_check": ROOT / "data/06_clean_road_networks_calculate_morphology_metrics/morphology_metrics_quality_checks.csv",
    "network_anomalies": ROOT / "data/06_clean_road_networks_calculate_morphology_metrics/road_network_anomaly_records.csv",
    "metrics_log": ROOT / "data/06_clean_road_networks_calculate_morphology_metrics/morphology_metrics_run_log.csv",
    "ohsome_wide": ROOT / "data/04_download_facilities_quality_history_data/ohsome/ohsome_quality_history_wide.csv",
    "ohsome_long": ROOT / "data/04_download_facilities_quality_history_data/ohsome/ohsome_quality_history_long.csv",
    "ohsome_status": ROOT / "data/04_download_facilities_quality_history_data/status/ohsome_history_status.csv",
    "overture_summary": ROOT / "data/04_download_facilities_quality_history_data/OverturePlaces/overture_places_summary.csv",
    "overture_status": ROOT / "data/04_download_facilities_quality_history_data/status/overture_places_status.csv",
    "osm_facilities_summary": ROOT / "data/04_download_facilities_quality_history_data/OSM_facilities/osm_facilities_extract_summary.csv",
    "osm_facilities_status": ROOT / "data/04_download_facilities_quality_history_data/status/osm_facilities_status.csv",
    "osm_facilities_parquet_dir": ROOT / "data/04_download_facilities_quality_history_data/OSM_facilities/extract_parquet",
}
OUTPUT_PATHS = {
    "quality_base_csv": OUT_DIR / "quality_base_metrics.csv",
    "quality_base_parquet": OUT_DIR / "quality_base_metrics.parquet",
    "city_quality_csv": OUT_DIR / "city_quality_scores.csv",
    "city_quality_parquet": OUT_DIR / "city_quality_scores.parquet",
    "local_quality_csv": OUT_DIR / "local_quality_flags.csv",
    "local_quality_parquet": OUT_DIR / "local_quality_flags.parquet",
    "review_csv": OUT_DIR / "quality_review_list.csv",
    "stats_csv": OUT_DIR / "quality_scores_summary_stats.csv",
    "execution_log_json": OUT_DIR / "quality_scores_run_log.json",
}

execution_log = {
    "step": "07_generate_quality_scores_type_confidence",
    "started_at": STARTED_AT,
    "python_version": sys.version,
    "platform": platform.platform(),
    "input_paths": {k: str(v) for k, v in INPUT_PATHS.items()},
    "output_paths": {k: str(v) for k, v in OUTPUT_PATHS.items()},
    "score_scale": SCORE_SCALE,
    "score_weights": SCORE_WEIGHTS,
    "tier_rules": TIER_RULES,
    "local_valid_area_min": LOCAL_VALID_AREA_MIN,
    "parquet_available": PARQUET_AVAILABLE,
    "notes": [
        "Step 7 only generates quality scores, quality weights, and quality_confidence_component.",
        "PCA, UMAP, clustering, morphotype assignment, scale stability, accessibility models, and final type_confidence are not executed.",
    ],
}

print("ROOT:", ROOT)
print("OUT_DIR:", OUT_DIR)
print("PARQUET_AVAILABLE:", PARQUET_AVAILABLE)
print("started_at:", STARTED_AT)


ROOT: /Volumes/ZHITAI2T/202606osm
OUT_DIR: /Volumes/ZHITAI2T/202606osm/data/07_generate_quality_scores_type_confidence
PARQUET_AVAILABLE: True
started_at: 2026-05-27T14:49:47


## Part 1: Read inputs and run integrity checks


In [2]:
# Part 1: Read inputs and run integrity checks
# Note: prefer Step 4 summary files; if a summary is missing, use status/long files as an in-memory fallback without writing back to the Step 04 directory.

def require_file(path: Path, label: str) -> None:
    """Confirm required input files exist; stop immediately if any are missing to avoid incomplete quality scores."""
    if not path.exists():
        raise FileNotFoundError(f"Missing required input {label}: {path}")


def read_csv_required(path: Path, label: str, **kwargs) -> pd.DataFrame:
    require_file(path, label)
    return pd.read_csv(path, low_memory=False, **kwargs)

# Read primary input tables.
city_master = read_csv_required(INPUT_PATHS["city_master"], "city_master")
population_built = read_csv_required(INPUT_PATHS["population_built"], "city_population_built_environment_metrics")
unit_quality = read_csv_required(INPUT_PATHS["unit_quality_check"], "spatial_units_quality_checks")
unit_index = read_csv_required(INPUT_PATHS["unit_index"], "spatial_units_index")
city_metrics = read_csv_required(INPUT_PATHS["city_metrics"], "city_morphology_metrics")
local_metrics = read_csv_required(INPUT_PATHS["local_metrics"], "local_morphology_metrics")
metrics_qc = read_csv_required(INPUT_PATHS["metrics_quality_check"], "morphology_metrics_quality_checks")
network_anomalies = read_csv_required(INPUT_PATHS["network_anomalies"], "road_network_anomaly_records")
metrics_log = read_csv_required(INPUT_PATHS["metrics_log"], "morphology_metrics_run_log")

# Read the OHSOME history wide table; if the wide table is missing, pivot the long table into an equivalent wide table.
if INPUT_PATHS["ohsome_wide"].exists():
    ohsome_wide = pd.read_csv(INPUT_PATHS["ohsome_wide"], low_memory=False)
elif INPUT_PATHS["ohsome_long"].exists():
    ohsome_long = pd.read_csv(INPUT_PATHS["ohsome_long"], low_memory=False)
    value_col = "value" if "value" in ohsome_long.columns else "count"
    ohsome_wide = (
        ohsome_long.pivot_table(
            index=["city_id", "city_name_en", "iso3", "timestamp"],
            columns="indicator",
            values=value_col,
            aggfunc="sum",
        )
        .reset_index()
        .rename_axis(None, axis=1)
    )
else:
    raise FileNotFoundError("Missing OHSOME wide/long history tables; cannot build historical_maturity_score.")

# If the Overture summary is missing, use the status table; both have the same structure in this project.
if INPUT_PATHS["overture_summary"].exists():
    overture_summary = pd.read_csv(INPUT_PATHS["overture_summary"], low_memory=False)
else:
    overture_summary = read_csv_required(INPUT_PATHS["overture_status"], "Overture status fallback")

# If the OSM facilities summary is missing, use the status table; this table is extract-level, and city-level counts are summarized from parquet in Part 2.
if INPUT_PATHS["osm_facilities_summary"].exists():
    osm_facilities_summary = pd.read_csv(INPUT_PATHS["osm_facilities_summary"], low_memory=False)
else:
    osm_facilities_summary = read_csv_required(INPUT_PATHS["osm_facilities_status"], "OSM facilities status fallback")

ohsome_status = read_csv_required(INPUT_PATHS["ohsome_status"], "OHSOME status")
overture_status = pd.read_csv(INPUT_PATHS["overture_status"], low_memory=False) if INPUT_PATHS["overture_status"].exists() else overture_summary.copy()
osm_facilities_status = pd.read_csv(INPUT_PATHS["osm_facilities_status"], low_memory=False) if INPUT_PATHS["osm_facilities_status"].exists() else osm_facilities_summary.copy()

# Boolean columns in CSV files can be read as strings; convert them consistently to booleans to avoid downstream filtering errors.
for df in [city_metrics, local_metrics]:
    if "valid_metric" in df.columns:
        df["valid_metric"] = df["valid_metric"].astype(str).str.lower().map({"true": True, "false": False}).fillna(df["valid_metric"].astype(bool))
    if "edge_unit" in df.columns:
        df["edge_unit"] = df["edge_unit"].astype(str).str.lower().map({"true": True, "false": False}).fillna(False).astype(bool)

if "edge_unit" in unit_index.columns:
    unit_index["edge_unit"] = unit_index["edge_unit"].astype(str).str.lower().map({"true": True, "false": False}).fillna(False).astype(bool)

# Structural integrity checks: row counts, uniqueness, sample groups, scales, and network types must match the upstream accepted state.
assert len(city_master) == 86, f"city_mastershould have 86 rows, actual {len(city_master)}"
assert city_master["city_id"].is_unique, "city_master has duplicate city_id values"
assert {"main_80", "china_pressure_test"}.issubset(set(city_master["sample_group"])), "sample groups main_80 and china_pressure_test are not both retained"
assert len(city_metrics) == 344, f"city_morphology_metricsshould have 344 rows, actual {len(city_metrics)}"
assert len(local_metrics) == 396_670, f"local_morphology_metricsshould have 396,670 rows, actual {len(local_metrics)}"

expected_city_scales = {"full_city", "core_5km"}
expected_local_scales = {"hex_1km", "hex_2km"}
expected_network_types = {"drive", "walk"}
assert set(city_metrics["network_type"].unique()) == expected_network_types, "city metrics are missing drive/walk coverage"
assert set(local_metrics["network_type"].unique()) == expected_network_types, "local metrics are missing drive/walk coverage"
assert set(city_metrics["scale"].unique()) == expected_city_scales, "city metrics are missing full_city/core_5km coverage"
assert set(local_metrics["scale"].unique()) == expected_local_scales, "local metrics are missing hex_1km/hex_2km coverage"

# Each city should have 2 city scales x 2 network_type values.
city_combo = city_metrics.groupby(["city_id", "scale", "network_type"]).size().reset_index(name="n")
expected_city_combo_count = len(city_master) * len(expected_city_scales) * len(expected_network_types)
assert len(city_combo) == expected_city_combo_count, f"city-scale combinations should be {expected_city_combo_count}, actual {len(city_combo)}"

# Each city should have 2 local scales x 2 network_type values, with at least one row per group.
local_combo = local_metrics.groupby(["city_id", "scale", "network_type"]).size().reset_index(name="n")
expected_local_combo_count = len(city_master) * len(expected_local_scales) * len(expected_network_types)
assert len(local_combo) == expected_local_combo_count, f"local-scale combinations should be {expected_local_combo_count}, actual {len(local_combo)}"
assert (local_combo["n"] > 0).all(), "at least one local-scale combination has zero rows"

# Step 4 input success status checks.
overture_success_count = int((overture_status["status"] == "success").sum()) if "status" in overture_status.columns else 0
osm_success_count = int((osm_facilities_status["status"] == "success").sum()) if "status" in osm_facilities_status.columns else 0
ohsome_success_count = int((ohsome_status["status"] == "success").sum()) if "status" in ohsome_status.columns else 0
assert overture_success_count == 86, f"Overture success should be 86, actual {overture_success_count}"
assert osm_success_count == 51, f"OSM facilities success should be 51, actual {osm_success_count}"
assert ohsome_success_count == 516, f"OHSOME success should be 516, actual {ohsome_success_count}"

input_row_counts = {
    "city_master": len(city_master),
    "population_built": len(population_built),
    "unit_quality": len(unit_quality),
    "unit_index": len(unit_index),
    "city_metrics": len(city_metrics),
    "local_metrics": len(local_metrics),
    "metrics_qc": len(metrics_qc),
    "network_anomalies": len(network_anomalies),
    "metrics_log": len(metrics_log),
    "ohsome_wide": len(ohsome_wide),
    "overture_summary": len(overture_summary),
    "osm_facilities_summary": len(osm_facilities_summary),
    "ohsome_status_success": ohsome_success_count,
    "overture_status_success": overture_success_count,
    "osm_facilities_status_success": osm_success_count,
}
execution_log["input_row_counts"] = input_row_counts
print(json.dumps(input_row_counts, ensure_ascii=False, indent=2))


{
  "city_master": 86,
  "population_built": 86,
  "unit_quality": 86,
  "unit_index": 198507,
  "city_metrics": 344,
  "local_metrics": 396670,
  "metrics_qc": 688,
  "network_anomalies": 1370,
  "metrics_log": 172,
  "ohsome_wide": 1634,
  "overture_summary": 86,
  "osm_facilities_summary": 51,
  "ohsome_status_success": 516,
  "overture_status_success": 86,
  "osm_facilities_status_success": 51
}


## Part 2: Build five quality base metric groups


In [3]:
# Part 2: Build five quality base metric groups
# Note: this part keeps interpretable raw base metrics; the total score is only composed in Part 4 using explicit weights.

def safe_divide(numerator, denominator):
    """Safe division: return NaN when the denominator is 0 or missing, preventing meaningless infinite values from entering scores."""
    numerator = pd.to_numeric(numerator, errors="coerce")
    denominator = pd.to_numeric(denominator, errors="coerce")
    return numerator.where(denominator != 0) / denominator.where(denominator != 0)


def score_higher_better(series: pd.Series, log1p: bool = False) -> pd.Series:
    """5%/95% robust normalization; larger values receive higher scores."""
    x = pd.to_numeric(series, errors="coerce").astype(float)
    if log1p:
        x = np.log1p(x.clip(lower=0))
    valid = x.dropna()
    out = pd.Series(np.nan, index=series.index, dtype="float64")
    if valid.empty:
        return out
    q05, q95 = valid.quantile([0.05, 0.95])
    if not np.isfinite(q05) or not np.isfinite(q95) or q95 <= q05:
        out.loc[x.notna()] = 100.0
        return out
    out = ((x.clip(q05, q95) - q05) / (q95 - q05) * 100).clip(0, 100)
    return out


def score_lower_better(series: pd.Series, log1p: bool = False) -> pd.Series:
    """5%/95% robust normalization; smaller values receive higher scores."""
    s = score_higher_better(series, log1p=log1p)
    return 100 - s


def score_ratio_closeness(series: pd.Series, max_factor: float = 3.0) -> pd.Series:
    """Near-one scoring for ratio metrics: 1 is best, and values near 1/max_factor or max_factor approach 0."""
    x = pd.to_numeric(series, errors="coerce").astype(float)
    out = pd.Series(np.nan, index=series.index, dtype="float64")
    valid_mask = x.gt(0) & np.isfinite(x)
    if not valid_mask.any():
        return out
    denom = math.log(max_factor)
    out.loc[valid_mask] = (100 * (1 - np.abs(np.log(x.loc[valid_mask])) / denom)).clip(0, 100)
    return out


def direct_share_score(series: pd.Series) -> pd.Series:
    """Map 0-1 share values directly to 0-100; clip out-of-range values."""
    x = pd.to_numeric(series, errors="coerce").astype(float)
    return (x.clip(0, 1) * 100).astype(float)


def weighted_average(score_frame: pd.DataFrame, weights: dict) -> pd.Series:
    """Compute a row-wise weighted mean using the given weights; when a subscore is missing, drop only its weight for that row."""
    numerator = pd.Series(0.0, index=score_frame.index)
    denominator = pd.Series(0.0, index=score_frame.index)
    for col, weight in weights.items():
        values = pd.to_numeric(score_frame[col], errors="coerce")
        mask = values.notna()
        numerator.loc[mask] += values.loc[mask] * weight
        denominator.loc[mask] += weight
    return numerator.where(denominator > 0) / denominator.where(denominator > 0)


def make_issue_note(row: pd.Series, flag_columns: list[str]) -> str:
    """Summarize False fields in input readiness status as readable notes."""
    missing = [col for col in flag_columns if not bool(row.get(col, False))]
    return ";".join(missing)

city_key_cols = ["city_id", "city_name_en", "country", "iso3", "region", "sample_group", "geofabrik_extract_id", "ucdb_pop_2025", "ucdb_area_km2"]
quality_base = city_master[city_key_cols].copy()

# ---------- Input readiness fields ----------
city_ids = set(city_master["city_id"])
population_ids = set(population_built["city_id"])
ohsome_ids = set(ohsome_wide["city_id"])
overture_success_by_city = overture_summary.set_index("city_id")["status"].eq("success") if "status" in overture_summary.columns else pd.Series(dtype=bool)
osm_extract_success = osm_facilities_summary.set_index("extract_id")["status"].eq("success") if "extract_id" in osm_facilities_summary.columns and "status" in osm_facilities_summary.columns else pd.Series(dtype=bool)

full_drive_valid = city_metrics.query("scale == 'full_city' and network_type == 'drive'").set_index("city_id")["valid_metric"].astype(bool)
full_walk_valid = city_metrics.query("scale == 'full_city' and network_type == 'walk'").set_index("city_id")["valid_metric"].astype(bool)
local_drive_present = local_metrics.query("network_type == 'drive'").groupby("city_id").size().gt(0)
local_walk_present = local_metrics.query("network_type == 'walk'").groupby("city_id").size().gt(0)

quality_base["has_city_master"] = quality_base["city_id"].isin(city_ids)
quality_base["has_population_built"] = quality_base["city_id"].isin(population_ids)
quality_base["has_ohsome_history"] = quality_base["city_id"].isin(ohsome_ids)
quality_base["has_overture_summary"] = quality_base["city_id"].map(overture_success_by_city).fillna(False).astype(bool)
quality_base["has_osm_facilities_summary"] = quality_base["geofabrik_extract_id"].map(osm_extract_success).fillna(False).astype(bool)
quality_base["has_city_metrics_drive"] = quality_base["city_id"].map(full_drive_valid).fillna(False).astype(bool)
quality_base["has_city_metrics_walk"] = quality_base["city_id"].map(full_walk_valid).fillna(False).astype(bool)
quality_base["has_local_metrics_drive"] = quality_base["city_id"].map(local_drive_present).fillna(False).astype(bool)
quality_base["has_local_metrics_walk"] = quality_base["city_id"].map(local_walk_present).fillna(False).astype(bool)
input_flag_cols = [
    "has_city_master",
    "has_population_built",
    "has_ohsome_history",
    "has_overture_summary",
    "has_osm_facilities_summary",
    "has_city_metrics_drive",
    "has_city_metrics_walk",
    "has_local_metrics_drive",
    "has_local_metrics_walk",
]
quality_base["input_ready"] = quality_base[input_flag_cols].all(axis=1)
quality_base["input_issue_note"] = quality_base.apply(lambda row: make_issue_note(row, input_flag_cols), axis=1)

# ---------- 1) Base metrics for network_integrity_score ----------
city_core_metrics = city_metrics[city_metrics["scale"].isin(["full_city", "core_5km"])].copy()
full_city_metrics = city_metrics[city_metrics["scale"].eq("full_city")].copy()

valid_rate_city_metrics = city_core_metrics.groupby("city_id")["valid_metric"].mean().rename("city_metric_valid_rate")
mean_giant_share = city_core_metrics.groupby("city_id")["giant_component_edge_share"].mean().rename("giant_component_edge_share_mean")
mean_component_count = city_core_metrics.groupby("city_id")["component_count"].mean().rename("component_count_mean")
full_edge_density_mean = full_city_metrics.groupby("city_id")["edge_density_km_per_km2"].mean().rename("full_city_edge_density_mean")
full_node_density_mean = full_city_metrics.groupby("city_id")["node_density_per_km2"].mean().rename("full_city_node_density_mean")
full_edge_count_sum = full_city_metrics.groupby("city_id")["edge_count"].sum().rename("full_city_edge_count_sum")
full_node_count_sum = full_city_metrics.groupby("city_id")["node_count"].sum().rename("full_city_node_count_sum")

full_flags = full_city_metrics.pivot_table(index="city_id", columns="network_type", values="valid_metric", aggfunc="max")
full_flags = full_flags.rename(columns={"drive": "full_city_drive_valid", "walk": "full_city_walk_valid"}).reset_index()
core_walk_valid = city_metrics.query("scale == 'core_5km' and network_type == 'walk'").set_index("city_id")["valid_metric"].rename("core_5km_walk_valid")
core_walk_edges = city_metrics.query("scale == 'core_5km' and network_type == 'walk'").set_index("city_id")["edge_count"].rename("core_5km_walk_edge_count")

qc_city_core = metrics_qc[metrics_qc["scale"].isin(["full_city", "core_5km"])].copy()
missing_cols = [
    "edge_density_km_per_km2_missing_rate",
    "node_density_per_km2_missing_rate",
    "orientation_entropy_missing_rate",
    "betweenness_gini_missing_rate",
]
qc_city_group = qc_city_core.groupby("city_id").agg(
    qc_row_count=("row_count", "sum"),
    qc_invalid_count=("invalid_count", "sum"),
    edge_density_km_per_km2_missing_rate=("edge_density_km_per_km2_missing_rate", "mean"),
    node_density_per_km2_missing_rate=("node_density_per_km2_missing_rate", "mean"),
    orientation_entropy_missing_rate=("orientation_entropy_missing_rate", "mean"),
    betweenness_gini_missing_rate=("betweenness_gini_missing_rate", "mean"),
).reset_index()
qc_city_group["invalid_metric_rate"] = safe_divide(qc_city_group["qc_invalid_count"], qc_city_group["qc_row_count"])
qc_city_group["core_metric_missing_rate_mean"] = qc_city_group[missing_cols].mean(axis=1)

severity_weight = {"info": 0.25, "warning": 1.0, "error": 2.0, "critical": 3.0}
network_anomalies["severity_weight"] = network_anomalies["severity"].map(severity_weight).fillna(1.0)
network_anomalies["affected_count_num"] = pd.to_numeric(network_anomalies.get("affected_count"), errors="coerce").fillna(0)
anomaly_group = network_anomalies.groupby("city_id").agg(
    anomaly_record_count=("anomaly_type", "size"),
    anomaly_weighted_count=("severity_weight", "sum"),
    anomaly_affected_count=("affected_count_num", "sum"),
).reset_index()
anomaly_type_counts = (
    network_anomalies.pivot_table(index="city_id", columns="anomaly_type", values="severity_weight", aggfunc="size", fill_value=0)
    .add_prefix("anomaly_type_count__")
    .reset_index()
)

network_base = quality_base[["city_id"]].merge(valid_rate_city_metrics, on="city_id", how="left")
for s in [mean_giant_share, mean_component_count, full_edge_density_mean, full_node_density_mean, full_edge_count_sum, full_node_count_sum, core_walk_valid, core_walk_edges]:
    network_base = network_base.merge(s.reset_index(), on="city_id", how="left")
network_base = network_base.merge(full_flags, on="city_id", how="left")
network_base = network_base.merge(qc_city_group, on="city_id", how="left")
network_base = network_base.merge(anomaly_group, on="city_id", how="left")
network_base = network_base.merge(anomaly_type_counts, on="city_id", how="left")
network_base["anomaly_record_count"] = network_base["anomaly_record_count"].fillna(0)
network_base["anomaly_weighted_count"] = network_base["anomaly_weighted_count"].fillna(0)
network_base["anomaly_affected_count"] = network_base["anomaly_affected_count"].fillna(0)
network_base["anomaly_affected_per_10k_edges"] = safe_divide(network_base["anomaly_affected_count"], network_base["full_city_edge_count_sum"]) * 10000
network_base["network_anomaly_pressure"] = np.log1p(network_base["anomaly_weighted_count"]) + np.log1p(network_base["anomaly_affected_per_10k_edges"].fillna(0))

# Invalid direction and detour ratios in morphology_metrics_run_log are used for review, not as opaque direct penalties.
log_group = metrics_log.groupby("city_id").agg(
    log_unique_edge_count=("unique_edge_count", "sum"),
    log_orientation_invalid_count=("orientation_invalid_count", "sum"),
    log_circuity_invalid_count=("circuity_invalid_count", "sum"),
    log_status_success_count=("status", lambda s: int((s == "success").sum())),
).reset_index()
log_group["orientation_invalid_rate"] = safe_divide(log_group["log_orientation_invalid_count"], log_group["log_unique_edge_count"])
log_group["circuity_invalid_rate"] = safe_divide(log_group["log_circuity_invalid_count"], log_group["log_unique_edge_count"])
network_base = network_base.merge(log_group, on="city_id", how="left")

# ---------- 2) Base metrics for historical_maturity_score ----------
ohsome = ohsome_wide.copy()
ohsome["year"] = pd.to_datetime(ohsome["timestamp"], errors="coerce").dt.year
indicator_cols = ["road_ways", "all_nodes", "all_ways", "building_ways", "poi_nodes", "named_poi_nodes"]
for col in indicator_cols:
    if col not in ohsome.columns:
        ohsome[col] = np.nan
    ohsome[col] = pd.to_numeric(ohsome[col], errors="coerce")

def ohsome_value_by_year(indicator: str, year: int) -> pd.Series:
    subset = ohsome[ohsome["year"].eq(year)].groupby("city_id")[indicator].max()
    subset.name = f"{indicator}_{year}"
    return subset

historical_base = quality_base[["city_id"]].copy()
for indicator in indicator_cols:
    for year in [2008, 2020, 2026]:
        historical_base = historical_base.merge(ohsome_value_by_year(indicator, year).reset_index(), on="city_id", how="left")

historical_base["road_ways_growth_2008_2026"] = safe_divide(
    historical_base["road_ways_2026"] - historical_base["road_ways_2008"],
    historical_base["road_ways_2008"].clip(lower=1),
)
historical_base["road_ways_recent_growth_2020_2026"] = safe_divide(
    historical_base["road_ways_2026"] - historical_base["road_ways_2020"],
    historical_base["road_ways_2020"].clip(lower=1),
)
historical_base["named_poi_share_2026"] = safe_divide(historical_base["named_poi_nodes_2026"], historical_base["poi_nodes_2026"])
historical_base["road_ways_stability_recent"] = (
    1 - np.abs(np.log(safe_divide(historical_base["road_ways_2026"] + 1, historical_base["road_ways_2020"] + 1))) / np.log(5)
).clip(0, 1)
last_nonzero = ohsome.loc[ohsome["road_ways"].fillna(0).gt(0)].groupby("city_id")["year"].max().rename("last_nonzero_year")
historical_base = historical_base.merge(last_nonzero.reset_index(), on="city_id", how="left")

# ---------- 3) Base metrics for poi_completeness_score ----------
# The Overture summary is city-level; the OSM facilities summary is extract-level, so city-level facility counts and naming rates are summarized precisely from extract parquet files here.
def build_osm_city_facility_summary(parquet_dir: Path) -> pd.DataFrame:
    rows = []
    parquet_files = sorted(parquet_dir.glob("*.parquet"))
    if not parquet_files:
        return pd.DataFrame(columns=["city_id", "osm_facility_record_count", "osm_named_facility_count", "osm_named_facility_ratio"])
    for pq_path in parquet_files:
        df = pd.read_parquet(pq_path, columns=["city_id", "name", "name_en"])
        if df.empty:
            continue
        name = df["name"].fillna("").astype(str).str.strip()
        name_en = df["name_en"].fillna("").astype(str).str.strip()
        df = df.assign(_named=name.ne("") | name_en.ne(""))
        grouped = df.groupby("city_id").agg(
            osm_facility_record_count=("city_id", "size"),
            osm_named_facility_count=("_named", "sum"),
        )
        rows.append(grouped)
    if not rows:
        return pd.DataFrame(columns=["city_id", "osm_facility_record_count", "osm_named_facility_count", "osm_named_facility_ratio"])
    result = pd.concat(rows).groupby(level=0).sum().reset_index()
    result["osm_named_facility_ratio"] = safe_divide(result["osm_named_facility_count"], result["osm_facility_record_count"])
    return result

osm_city_facilities = build_osm_city_facility_summary(INPUT_PATHS["osm_facilities_parquet_dir"])
overture_city = overture_summary[["city_id", "record_count"]].rename(columns={"record_count": "overture_record_count"}).copy()
overture_city["overture_record_count"] = pd.to_numeric(overture_city["overture_record_count"], errors="coerce")

poi_base = quality_base[["city_id", "ucdb_pop_2025"]].merge(
    population_built[["city_id", "ghsl_pop_2020_100m_sum"]], on="city_id", how="left"
)
poi_base = poi_base.merge(overture_city, on="city_id", how="left")
poi_base = poi_base.merge(osm_city_facilities, on="city_id", how="left")
for col in ["overture_record_count", "osm_facility_record_count", "osm_named_facility_count"]:
    poi_base[col] = pd.to_numeric(poi_base[col], errors="coerce").fillna(0)
poi_base["poi_population_denominator"] = poi_base["ghsl_pop_2020_100m_sum"].fillna(poi_base["ucdb_pop_2025"])
poi_base["overture_places_per_100k_pop"] = safe_divide(poi_base["overture_record_count"], poi_base["poi_population_denominator"]) * 100000
poi_base["osm_facilities_per_100k_pop"] = safe_divide(poi_base["osm_facility_record_count"], poi_base["poi_population_denominator"]) * 100000
poi_base["poi_source_balance"] = safe_divide(
    np.minimum(poi_base["overture_record_count"], poi_base["osm_facility_record_count"]),
    np.maximum(poi_base["overture_record_count"], poi_base["osm_facility_record_count"]),
)
poi_base["named_poi_ratio"] = poi_base["osm_named_facility_ratio"]

# ---------- 4) Base metrics for population_built_support_score ----------
pop_cols = [
    "city_id",
    "ghsl_pop_2020_100m_sum",
    "worldpop_pop_2020_sum",
    "worldpop_to_ghsl_pop_2020_100m_ratio",
    "ghsl_pop_2020_100m_to_ucdb_pop_2025_ratio",
    "ghsl_built_s_2020_100m_mean",
    "ghsl_built_share_2020_100m",
    "ghsl_pop_2020_100m_valid_pixel_count",
    "ghsl_built_s_2020_100m_valid_pixel_count",
    "worldpop_pop_2020_valid_pixel_count",
]
population_base = population_built[pop_cols].copy()
population_base["raster_valid_pixel_count"] = population_base[
    ["ghsl_pop_2020_100m_valid_pixel_count", "ghsl_built_s_2020_100m_valid_pixel_count", "worldpop_pop_2020_valid_pixel_count"]
].min(axis=1)
unit_support = unit_quality[[
    "city_id",
    "full_area_coverage_ratio",
    "core_area_coverage_ratio",
    "hex1_area_coverage_ratio",
    "hex2_area_coverage_ratio",
    "full_invalid_geometry_count",
    "core_invalid_geometry_count",
    "hex1_invalid_geometry_count",
    "hex2_invalid_geometry_count",
    "full_count",
    "core_count",
    "hex1_count",
    "hex2_count",
    "full_built_share_mean",
]].copy()
unit_support["unit_area_coverage_mean"] = unit_support[["full_area_coverage_ratio", "core_area_coverage_ratio", "hex1_area_coverage_ratio", "hex2_area_coverage_ratio"]].mean(axis=1)
unit_support["unit_invalid_geometry_total"] = unit_support[["full_invalid_geometry_count", "core_invalid_geometry_count", "hex1_invalid_geometry_count", "hex2_invalid_geometry_count"]].sum(axis=1)
unit_support["unit_count_total"] = unit_support[["full_count", "core_count", "hex1_count", "hex2_count"]].sum(axis=1)
unit_support["unit_invalid_geometry_rate"] = safe_divide(unit_support["unit_invalid_geometry_total"], unit_support["unit_count_total"])
population_base = population_base.merge(unit_support[["city_id", "unit_area_coverage_mean", "unit_invalid_geometry_rate", "full_built_share_mean"]], on="city_id", how="left")
population_base["built_share_mean"] = population_base["ghsl_built_share_2020_100m"].fillna(population_base["full_built_share_mean"])

# ---------- 5) Base metrics for local_coverage_score ----------
local_qc = metrics_qc[metrics_qc["scale"].isin(["hex_1km", "hex_2km"])].copy()
local_valid_pivot = local_qc.pivot_table(index="city_id", columns=["scale", "network_type"], values="valid_ratio", aggfunc="mean")
local_valid_pivot.columns = [f"{scale}_valid_ratio_{network}" for scale, network in local_valid_pivot.columns]
local_valid_pivot = local_valid_pivot.reset_index()

local_tmp = local_metrics[["city_id", "scale", "network_type", "valid_metric", "edge_unit", "valid_area_ratio", "edge_count", "node_count", "total_edge_length_km"]].copy()
local_tmp["local_empty_unit"] = (pd.to_numeric(local_tmp["edge_count"], errors="coerce").fillna(0) <= 0) | (pd.to_numeric(local_tmp["total_edge_length_km"], errors="coerce").fillna(0) <= 0)
local_tmp["local_invalid_unit"] = ~local_tmp["valid_metric"].astype(bool)
local_tmp["local_boundary_artifact_raw"] = local_tmp["edge_unit"] & (local_tmp["valid_area_ratio"] < local_tmp["scale"].map(LOCAL_VALID_AREA_MIN).fillna(0))
local_group = local_tmp.groupby("city_id").agg(
    local_unit_row_count=("city_id", "size"),
    valid_local_unit_count=("valid_metric", "sum"),
    empty_unit_share=("local_empty_unit", "mean"),
    invalid_local_unit_rate=("local_invalid_unit", "mean"),
    boundary_artifact_rate=("local_boundary_artifact_raw", "mean"),
).reset_index()
local_coverage_base = quality_base[["city_id"]].merge(local_valid_pivot, on="city_id", how="left").merge(local_group, on="city_id", how="left")
valid_ratio_cols = ["hex_1km_valid_ratio_drive", "hex_1km_valid_ratio_walk", "hex_2km_valid_ratio_drive", "hex_2km_valid_ratio_walk"]
local_coverage_base["local_valid_ratio_mean"] = local_coverage_base[valid_ratio_cols].mean(axis=1)
local_coverage_base["local_valid_ratio_min"] = local_coverage_base[valid_ratio_cols].min(axis=1)

# Merge all base metrics.
for part in [network_base, historical_base, poi_base, population_base, local_coverage_base]:
    duplicate_cols = [c for c in part.columns if c in quality_base.columns and c != "city_id"]
    quality_base = quality_base.merge(part.drop(columns=duplicate_cols), on="city_id", how="left")

print("quality_base columns:", len(quality_base.columns))
print("OSM city facilities covered:", int(osm_city_facilities["city_id"].nunique()), "cities")
print("input_ready counts:")
print(quality_base["input_ready"].value_counts(dropna=False).to_string())


quality_base columns: 113
OSM city facilities covered: 86 cities
input_ready counts:
input_ready
True    86


## Part 3: Standardize quality subscores


In [4]:
# Part 3: Standardize quality subscores
# Note: all subscores are output on a 0-100 scale; robust normalization is used, with explicit penalties for missingness, anomalies, and very low coverage.

# 1) Road-network structural integrity: full-city/core availability, connectivity, density support, missingness, and anomaly pressure.
network_scores = pd.DataFrame(index=quality_base.index)
network_scores["network_valid_metric_score"] = direct_share_score(1 - quality_base["invalid_metric_rate"].fillna(1))
network_scores["network_giant_component_score"] = direct_share_score(quality_base["giant_component_edge_share_mean"])
network_scores["network_component_count_score"] = score_lower_better(quality_base["component_count_mean"])
network_scores["network_edge_density_score"] = score_higher_better(quality_base["full_city_edge_density_mean"], log1p=True)
network_scores["network_node_density_score"] = score_higher_better(quality_base["full_city_node_density_mean"], log1p=True)
network_scores["network_missing_metric_score"] = direct_share_score(1 - quality_base["core_metric_missing_rate_mean"].fillna(1))
network_scores["network_anomaly_score"] = score_lower_better(quality_base["network_anomaly_pressure"].fillna(0), log1p=False)
network_scores["network_density_score"] = network_scores[["network_edge_density_score", "network_node_density_score"]].mean(axis=1)
quality_base["network_integrity_score"] = weighted_average(
    network_scores,
    {
        "network_valid_metric_score": 0.25,
        "network_giant_component_score": 0.15,
        "network_component_count_score": 0.10,
        "network_density_score": 0.20,
        "network_missing_metric_score": 0.15,
        "network_anomaly_score": 0.15,
    },
).clip(0, 100)

# 2) OSM historical maturity: current road/node/building/POI volume, named POI share, recent stability, and last nonzero year.
historical_scores = pd.DataFrame(index=quality_base.index)
historical_scores["hist_road_ways_2026_score"] = score_higher_better(quality_base["road_ways_2026"], log1p=True)
historical_scores["hist_all_nodes_2026_score"] = score_higher_better(quality_base["all_nodes_2026"], log1p=True)
historical_scores["hist_all_ways_2026_score"] = score_higher_better(quality_base["all_ways_2026"], log1p=True)
historical_scores["hist_building_ways_2026_score"] = score_higher_better(quality_base["building_ways_2026"], log1p=True)
historical_scores["hist_poi_nodes_2026_score"] = score_higher_better(quality_base["poi_nodes_2026"], log1p=True)
historical_scores["hist_named_poi_share_score"] = direct_share_score(quality_base["named_poi_share_2026"])
historical_scores["hist_stability_recent_score"] = direct_share_score(quality_base["road_ways_stability_recent"])
historical_scores["hist_last_nonzero_year_score"] = ((quality_base["last_nonzero_year"] - 2008) / (2026 - 2008) * 100).clip(0, 100)
historical_scores["hist_volume_score"] = historical_scores[
    ["hist_all_nodes_2026_score", "hist_all_ways_2026_score", "hist_building_ways_2026_score", "hist_poi_nodes_2026_score"]
].mean(axis=1)
quality_base["historical_maturity_score"] = weighted_average(
    historical_scores,
    {
        "hist_road_ways_2026_score": 0.25,
        "hist_volume_score": 0.35,
        "hist_named_poi_share_score": 0.10,
        "hist_stability_recent_score": 0.20,
        "hist_last_nonzero_year_score": 0.10,
    },
).clip(0, 100)

# 3) Facility completeness: Overture/OSM dual-source facilities per 100,000 people, source balance, and OSM naming rate.
poi_scores = pd.DataFrame(index=quality_base.index)
poi_scores["poi_overture_density_score"] = score_higher_better(quality_base["overture_places_per_100k_pop"], log1p=True)
poi_scores["poi_osm_density_score"] = score_higher_better(quality_base["osm_facilities_per_100k_pop"], log1p=True)
poi_scores["poi_source_balance_score"] = direct_share_score(quality_base["poi_source_balance"].fillna(0))
poi_scores["poi_named_ratio_score"] = direct_share_score(quality_base["named_poi_ratio"].fillna(0))
poi_scores["poi_summary_status_score"] = direct_share_score(quality_base["has_overture_summary"].astype(float) * quality_base["has_osm_facilities_summary"].astype(float))
quality_base["poi_completeness_score"] = weighted_average(
    poi_scores,
    {
        "poi_overture_density_score": 0.35,
        "poi_osm_density_score": 0.35,
        "poi_source_balance_score": 0.15,
        "poi_named_ratio_score": 0.10,
        "poi_summary_status_score": 0.05,
    },
).clip(0, 100)

# 4) Population and built-environment support: population-source consistency, UCDB alignment, valid raster pixels, built-up-area support, and spatial_units coverage.
pop_scores = pd.DataFrame(index=quality_base.index)
pop_scores["pop_worldpop_ghsl_ratio_score"] = score_ratio_closeness(quality_base["worldpop_to_ghsl_pop_2020_100m_ratio"], max_factor=3.0)
pop_scores["pop_ghsl_ucdb_ratio_score"] = score_ratio_closeness(quality_base["ghsl_pop_2020_100m_to_ucdb_pop_2025_ratio"], max_factor=3.0)
pop_scores["pop_raster_valid_pixel_score"] = score_higher_better(quality_base["raster_valid_pixel_count"], log1p=True)
pop_scores["pop_built_share_score"] = score_higher_better(quality_base["built_share_mean"], log1p=False)
pop_scores["pop_unit_coverage_score"] = (
    direct_share_score(quality_base["unit_area_coverage_mean"].fillna(0)) * 0.70
    + direct_share_score(1 - quality_base["unit_invalid_geometry_rate"].fillna(1)) * 0.30
)
quality_base["population_built_support_score"] = weighted_average(
    pop_scores,
    {
        "pop_worldpop_ghsl_ratio_score": 0.25,
        "pop_ghsl_ucdb_ratio_score": 0.25,
        "pop_raster_valid_pixel_score": 0.20,
        "pop_built_share_score": 0.15,
        "pop_unit_coverage_score": 0.15,
    },
).clip(0, 100)

# 5) Local-scale coverage: four local valid_ratio values, empty-unit rate, invalid-unit rate, valid-unit count, and boundary-artifact rate.
local_scores = pd.DataFrame(index=quality_base.index)
local_scores["local_valid_ratio_mean_score"] = direct_share_score(quality_base["local_valid_ratio_mean"])
local_scores["local_valid_ratio_min_score"] = direct_share_score(quality_base["local_valid_ratio_min"])
local_scores["local_empty_unit_score"] = direct_share_score(1 - quality_base["empty_unit_share"].fillna(1))
local_scores["local_invalid_unit_score"] = direct_share_score(1 - quality_base["invalid_local_unit_rate"].fillna(1))
local_scores["local_valid_unit_count_score"] = score_higher_better(quality_base["valid_local_unit_count"], log1p=True)
local_scores["local_boundary_artifact_score"] = direct_share_score(1 - quality_base["boundary_artifact_rate"].fillna(1))
quality_base["local_coverage_score"] = weighted_average(
    local_scores,
    {
        "local_valid_ratio_mean_score": 0.40,
        "local_valid_ratio_min_score": 0.15,
        "local_empty_unit_score": 0.15,
        "local_invalid_unit_score": 0.15,
        "local_valid_unit_count_score": 0.10,
        "local_boundary_artifact_score": 0.05,
    },
).clip(0, 100)

component_cols = list(SCORE_WEIGHTS.keys())
for col in component_cols:
    missing = int(quality_base[col].isna().sum())
    if missing:
        execution_log.setdefault("warnings", []).append(f"{col} has {missing} missing values; they were treated as score 0 in the total score.")
    quality_base[col] = quality_base[col].fillna(0).clip(0, 100)

print(quality_base[["city_id", *component_cols]].describe().T[["mean", "min", "max"]])


                                     mean        min        max
network_integrity_score         74.525761  46.875000  87.658693
historical_maturity_score       60.612987  25.927385  91.978843
poi_completeness_score          61.253227  14.807071  89.375843
population_built_support_score  75.700803  58.327961  96.628083
local_coverage_score            84.855534  44.874326  96.046595


## Part 4: City quality total score


In [5]:
# Part 4: City quality total score
# Note: compose quality_score using the first-version weights specified by the user and generate quality_confidence_component.
quality_base["quality_score"] = sum(quality_base[col] * weight for col, weight in SCORE_WEIGHTS.items()).clip(0, 100)
quality_base["quality_confidence_component"] = (quality_base["quality_score"] / 100).clip(0, 1)

# First build candidate review reasons; Part 5 converts them into tiers.
quality_base["full_city_core_network_ready"] = quality_base["full_city_drive_valid"].fillna(False).astype(bool) & quality_base["full_city_walk_valid"].fillna(False).astype(bool)
quality_base["severe_input_missing"] = ~quality_base["input_ready"].fillna(False).astype(bool)
quality_base["local_valid_ratio_low_for_tier"] = quality_base["local_valid_ratio_min"].lt(0.75).fillna(True)
quality_base["poi_weak_for_tier"] = quality_base["poi_completeness_score"].lt(35)
quality_base["history_weak_for_tier"] = quality_base["historical_maturity_score"].lt(35)
quality_base["population_weak_for_tier"] = quality_base["population_built_support_score"].lt(35)
quality_base["network_weak_for_tier"] = quality_base["network_integrity_score"].lt(55)

reason_rules = [
    ("score_below_50", quality_base["quality_score"].lt(50)),
    ("score_between_50_and_70", quality_base["quality_score"].ge(50) & quality_base["quality_score"].lt(70)),
    ("full_city_drive_or_walk_unavailable", ~quality_base["full_city_core_network_ready"]),
    ("severe_input_missing", quality_base["severe_input_missing"]),
    ("local_valid_ratio_below_0.75", quality_base["local_valid_ratio_low_for_tier"]),
    ("poi_score_below_35", quality_base["poi_weak_for_tier"]),
    ("historical_score_below_35", quality_base["history_weak_for_tier"]),
    ("population_support_score_below_35", quality_base["population_weak_for_tier"]),
    ("network_integrity_score_below_55", quality_base["network_weak_for_tier"]),
]
review_reasons = []
for idx, row in quality_base.iterrows():
    reasons = [name for name, mask in reason_rules if bool(mask.loc[idx])]
    review_reasons.append(";".join(reasons))
quality_base["review_reason"] = review_reasons
quality_base["review_flag"] = quality_base["review_reason"].ne("")

print(quality_base["quality_score"].describe())


count    86.000000
mean     71.240146
std       7.520531
min      51.554208
25%      65.776126
50%      71.834648
75%      76.137547
max      85.988186
Name: quality_score, dtype: float64


## Part 5: Assign quality_tier


In [6]:
# Part 5: Assign quality_tier
# Note: use a score plus hard-issue rule, not tertiles; excluded_candidate does not remove cities.
def assign_quality_tier(row: pd.Series) -> str:
    if (
        row["quality_score"] < 50
        or not bool(row["full_city_core_network_ready"])
        or bool(row["severe_input_missing"])
        or bool(row["network_weak_for_tier"])
    ):
        return "excluded_candidate"
    if (
        row["quality_score"] < 70
        or bool(row["local_valid_ratio_low_for_tier"])
        or bool(row["poi_weak_for_tier"])
        or bool(row["history_weak_for_tier"])
        or bool(row["population_weak_for_tier"])
    ):
        return "sensitivity"
    return "core"

quality_base["quality_tier"] = quality_base.apply(assign_quality_tier, axis=1)
assert set(quality_base["quality_tier"].unique()).issubset({"core", "sensitivity", "excluded_candidate"})
print(quality_base["quality_tier"].value_counts().to_string())


quality_tier
core                  48
sensitivity           37
excluded_candidate     1


## Part 6: Generate quality_weight


In [7]:
# Part 6: Generate quality_weight
# Note: core is fixed at 1; sensitivity/excluded_candidate are continuously mapped from quality_score and clipped to 0.25-1.00.
def map_quality_weight(row: pd.Series) -> float:
    score = float(row["quality_score"])
    tier = row["quality_tier"]
    if tier == "core":
        return 1.00
    if tier == "sensitivity":
        return float(np.clip(0.60 + ((score - 50) / 20) * 0.30, 0.60, 0.90))
    return float(np.clip(0.25 + (score / 50) * 0.25, 0.25, 0.50))

quality_base["quality_weight"] = quality_base.apply(map_quality_weight, axis=1).round(4)
assert quality_base["quality_weight"].between(0.25, 1.00).all(), "quality_weight is outside 0.25-1.00"
print(quality_base.groupby("quality_tier")["quality_weight"].describe().to_string())


                    count      mean       std     min     25%    50%    75%  max
quality_tier                                                                    
core                 48.0  1.000000  0.000000  1.0000  1.0000  1.000  1.000  1.0
excluded_candidate    1.0  0.500000       NaN  0.5000  0.5000  0.500  0.500  0.5
sensitivity          37.0  0.820262  0.065087  0.6233  0.7972  0.831  0.873  0.9


## Part 7: Local quality flags


In [8]:
# Part 7: Local quality flags
# Note: empty grid cells are not automatically errors, but they do not enter local morphology clustering; very small boundary slices and anomalously high density are treated as boundary_artifact.
local_quality = local_metrics[[
    "city_id",
    "unit_id",
    "network_type",
    "scale",
    "valid_metric",
    "valid_area_ratio",
    "edge_unit",
    "edge_count",
    "node_count",
    "total_edge_length_km",
    "edge_density_km_per_km2",
    "node_density_per_km2",
    "edge_circuity_mean",
    "orientation_entropy",
    "betweenness_gini",
    "validity_note",
]].copy()

for col in ["valid_area_ratio", "edge_count", "node_count", "total_edge_length_km", "edge_density_km_per_km2", "node_density_per_km2", "edge_circuity_mean", "orientation_entropy", "betweenness_gini"]:
    local_quality[col] = pd.to_numeric(local_quality[col], errors="coerce")
local_quality["valid_metric"] = local_quality["valid_metric"].astype(bool)
local_quality["edge_unit"] = local_quality["edge_unit"].astype(bool)

# Use the 99.5th percentile within each scale/network to flag anomalously high density, avoiding hard-coded treatment of genuinely high-density cities as anomalies.
density_thresholds = (
    local_quality.loc[local_quality["valid_metric"]]
    .groupby(["scale", "network_type"])[["edge_density_km_per_km2", "node_density_per_km2"]]
    .quantile(0.995)
    .rename(columns={"edge_density_km_per_km2": "edge_density_p995", "node_density_per_km2": "node_density_p995"})
    .reset_index()
)
local_quality = local_quality.merge(density_thresholds, on=["scale", "network_type"], how="left")
local_quality["min_valid_area_ratio"] = local_quality["scale"].map(LOCAL_VALID_AREA_MIN).fillna(0)

local_quality["local_empty_unit_flag"] = local_quality["edge_count"].fillna(0).le(0) | local_quality["total_edge_length_km"].fillna(0).le(0)
local_quality["local_low_edge_count_flag"] = local_quality["edge_count"].fillna(0).lt(3)
local_quality["local_low_node_count_flag"] = local_quality["node_count"].fillna(0).lt(2)
local_quality["local_high_density_flag"] = (
    (local_quality["edge_density_km_per_km2"] > local_quality["edge_density_p995"])
    | (local_quality["node_density_per_km2"] > local_quality["node_density_p995"])
).fillna(False)
local_quality["local_circuity_invalid_flag"] = (~local_quality["local_empty_unit_flag"]) & (
    local_quality["edge_circuity_mean"].isna() | local_quality["edge_circuity_mean"].lt(1) | local_quality["edge_circuity_mean"].gt(3)
)
local_quality["local_orientation_invalid_flag"] = (~local_quality["local_empty_unit_flag"]) & (
    local_quality["orientation_entropy"].isna() | local_quality["orientation_entropy"].lt(0) | local_quality["orientation_entropy"].gt(3.60)
)
local_quality["boundary_artifact_condition"] = (
    local_quality["edge_unit"]
    & local_quality["valid_area_ratio"].lt(local_quality["min_valid_area_ratio"])
) | (
    local_quality["local_high_density_flag"] & local_quality["valid_area_ratio"].lt(0.50)
)

# The score is a local row-level usability score; it does not enter the city total score and only supports downstream local signature filtering.
penalty = pd.Series(0.0, index=local_quality.index)
penalty += (~local_quality["valid_metric"]).astype(float) * 45
penalty += local_quality["local_empty_unit_flag"].astype(float) * 30
penalty += local_quality["local_low_edge_count_flag"].astype(float) * 15
penalty += local_quality["local_low_node_count_flag"].astype(float) * 15
penalty += local_quality["local_high_density_flag"].astype(float) * 20
penalty += local_quality["local_circuity_invalid_flag"].astype(float) * 15
penalty += local_quality["local_orientation_invalid_flag"].astype(float) * 15
penalty += local_quality["boundary_artifact_condition"].astype(float) * 25
local_quality["local_metric_validity_score"] = (100 - penalty).clip(0, 100).round(2)

def assign_local_flag(row: pd.Series) -> str:
    if bool(row["boundary_artifact_condition"]):
        return "boundary_artifact"
    if bool(row["valid_metric"]):
        return "valid"
    if bool(row["local_empty_unit_flag"]) or bool(row["local_low_edge_count_flag"]) or bool(row["local_low_node_count_flag"]):
        return "sparse"
    return "invalid_metric"

local_quality["local_quality_flag"] = local_quality.apply(assign_local_flag, axis=1)

reason_cols = [
    ("empty_unit", "local_empty_unit_flag"),
    ("low_edge_count", "local_low_edge_count_flag"),
    ("low_node_count", "local_low_node_count_flag"),
    ("high_density", "local_high_density_flag"),
    ("circuity_invalid", "local_circuity_invalid_flag"),
    ("orientation_invalid", "local_orientation_invalid_flag"),
    ("boundary_artifact", "boundary_artifact_condition"),
    ("invalid_metric", None),
]
reasons = []
for _, row in local_quality.iterrows():
    parts = []
    for label, col in reason_cols:
        if col is None:
            if not bool(row["valid_metric"]):
                parts.append(label)
        elif bool(row[col]):
            parts.append(label)
    if isinstance(row.get("validity_note"), str) and row["validity_note"].strip():
        parts.append(row["validity_note"].strip())
    reasons.append(";".join(dict.fromkeys(parts)) if parts else "valid")
local_quality["local_quality_reason"] = reasons

local_quality["use_in_local_signature"] = (
    local_quality["valid_metric"]
    & local_quality["edge_count"].gt(0)
    & local_quality["node_count"].gt(0)
    & local_quality["total_edge_length_km"].gt(0)
    & local_quality["valid_area_ratio"].ge(local_quality["min_valid_area_ratio"])
)
local_quality["use_in_local_clustering"] = (
    local_quality["use_in_local_signature"]
    & ~local_quality["boundary_artifact_condition"]
    & ~local_quality["local_high_density_flag"]
    & ~local_quality["local_circuity_invalid_flag"]
    & ~local_quality["local_orientation_invalid_flag"]
)

local_quality_output_cols = [
    "city_id",
    "unit_id",
    "network_type",
    "scale",
    "valid_metric",
    "valid_area_ratio",
    "edge_count",
    "node_count",
    "total_edge_length_km",
    "local_metric_validity_score",
    "local_empty_unit_flag",
    "local_low_edge_count_flag",
    "local_low_node_count_flag",
    "local_high_density_flag",
    "local_circuity_invalid_flag",
    "local_orientation_invalid_flag",
    "local_quality_flag",
    "local_quality_reason",
    "use_in_local_signature",
    "use_in_local_clustering",
]
local_quality = local_quality[local_quality_output_cols].copy()
print("local_quality rows:", len(local_quality))
print(local_quality["local_quality_flag"].value_counts(dropna=False).to_string())
print("use_in_local_signature:", int(local_quality["use_in_local_signature"].sum()))
print("use_in_local_clustering:", int(local_quality["use_in_local_clustering"].sum()))


local_quality rows: 396670
local_quality_flag
valid                336479
sparse                35857
boundary_artifact     20481
invalid_metric         3853
use_in_local_signature: 336544
use_in_local_clustering: 333547


## Part 8: Quality review list


In [9]:
# Part 8: Quality review list
# Note: the review list is an entry point for manual QA and sensitivity analysis; it does not remove any city.
review_rows = []
city_meta = quality_base.set_index("city_id")[["city_name_en", "region", "sample_group"]]

def add_review(city_id, issue_type, severity, evidence_value, suggested_action, related_network_type="", related_scale=""):
    meta = city_meta.loc[city_id]
    review_rows.append(
        {
            "city_id": city_id,
            "city_name_en": meta["city_name_en"],
            "region": meta["region"],
            "sample_group": meta["sample_group"],
            "issue_type": issue_type,
            "issue_severity": severity,
            "related_network_type": related_network_type,
            "related_scale": related_scale,
            "evidence_value": evidence_value,
            "suggested_action": suggested_action,
        }
    )

# Thresholds combine distribution-based rules with domain-interpretable hard limits.
anomaly_pressure_p90 = quality_base["network_anomaly_pressure"].quantile(0.90)
anomaly_affected_rate_p90 = quality_base["anomaly_affected_per_10k_edges"].quantile(0.90)
orientation_rate_p95 = quality_base["orientation_invalid_rate"].quantile(0.95)

for _, row in quality_base.iterrows():
    cid = row["city_id"]
    if row["quality_score"] < 50:
        add_review(cid, "low_quality_score", "high", round(row["quality_score"], 3), "exclude_after_review")

    for col in ["hex_1km_valid_ratio_drive", "hex_1km_valid_ratio_walk", "hex_2km_valid_ratio_drive", "hex_2km_valid_ratio_walk"]:
        value = row.get(col)
        if pd.notna(value) and value < 0.75:
            scale = "hex_1km" if col.startswith("hex_1km") else "hex_2km"
            network_type = "drive" if col.endswith("drive") else "walk"
            severity = "high" if value < 0.50 else "medium"
            add_review(cid, "low_local_valid_ratio", severity, f"{col}={value:.3f}", "keep_sensitivity_only", network_type, scale)

    if (not bool(row.get("core_5km_walk_valid", False))) or pd.to_numeric(row.get("core_5km_walk_edge_count"), errors="coerce") <= 0:
        add_review(cid, "walk_core_empty_or_invalid", "high", f"valid={row.get('core_5km_walk_valid')}; edge_count={row.get('core_5km_walk_edge_count')}", "manual_network_check", "walk", "core_5km")

    if row["network_anomaly_pressure"] >= anomaly_pressure_p90 or row["anomaly_affected_per_10k_edges"] >= anomaly_affected_rate_p90:
        add_review(cid, "high_anomaly_count", "medium", f"pressure={row['network_anomaly_pressure']:.3f}; affected_per_10k_edges={row['anomaly_affected_per_10k_edges']:.3f}", "manual_network_check")

    if (pd.notna(row["road_ways_stability_recent"]) and row["road_ways_stability_recent"] < 0.40) or (pd.notna(row["road_ways_recent_growth_2020_2026"]) and row["road_ways_recent_growth_2020_2026"] > 3):
        add_review(cid, "ohsome_history_unstable", "medium", f"recent_growth={row['road_ways_recent_growth_2020_2026']:.3f}; stability={row['road_ways_stability_recent']:.3f}", "keep_sensitivity_only")

    if row["overture_places_per_100k_pop"] < quality_base["overture_places_per_100k_pop"].quantile(0.10) and row["osm_facilities_per_100k_pop"] < quality_base["osm_facilities_per_100k_pop"].quantile(0.10):
        add_review(cid, "poi_extremely_sparse", "medium", f"overture_per_100k={row['overture_places_per_100k_pop']:.3f}; osm_per_100k={row['osm_facilities_per_100k_pop']:.3f}", "keep_sensitivity_only")

    wp_ratio = row.get("worldpop_to_ghsl_pop_2020_100m_ratio")
    ghsl_ucdb_ratio = row.get("ghsl_pop_2020_100m_to_ucdb_pop_2025_ratio")
    if (pd.notna(wp_ratio) and (wp_ratio < 0.50 or wp_ratio > 2.00)) or (pd.notna(ghsl_ucdb_ratio) and (ghsl_ucdb_ratio < 0.50 or ghsl_ucdb_ratio > 1.75)):
        add_review(cid, "population_source_mismatch", "medium", f"worldpop_ghsl={wp_ratio:.3f}; ghsl_ucdb={ghsl_ucdb_ratio:.3f}", "manual_boundary_check")

    if (pd.notna(row["giant_component_edge_share_mean"]) and row["giant_component_edge_share_mean"] < 0.98) or (pd.notna(row["component_count_mean"]) and row["component_count_mean"] > 1.5):
        add_review(cid, "network_disconnected", "medium", f"giant_share={row['giant_component_edge_share_mean']:.3f}; component_count={row['component_count_mean']:.3f}", "manual_network_check")

    if pd.notna(row.get("orientation_invalid_rate")) and row["orientation_invalid_rate"] >= orientation_rate_p95 and row["orientation_invalid_rate"] > 0.005:
        add_review(cid, "orientation_unavailable", "medium", f"orientation_invalid_rate={row['orientation_invalid_rate']:.5f}", "manual_network_check")

    if pd.notna(row.get("boundary_artifact_rate")) and row["boundary_artifact_rate"] > 0.05:
        add_review(cid, "boundary_or_tiny_area_issue", "medium", f"boundary_artifact_rate={row['boundary_artifact_rate']:.3f}", "manual_boundary_check")

review_columns = [
    "city_id",
    "city_name_en",
    "region",
    "sample_group",
    "issue_type",
    "issue_severity",
    "related_network_type",
    "related_scale",
    "evidence_value",
    "suggested_action",
]
review_list = pd.DataFrame(review_rows, columns=review_columns)
print("review rows:", len(review_list))
if len(review_list):
    print(review_list["issue_type"].value_counts().to_string())


review rows: 103
issue_type
boundary_or_tiny_area_issue    58
low_local_valid_ratio          20
high_anomaly_count             10
ohsome_history_unstable         6
orientation_unavailable         5
walk_core_empty_or_invalid      2
poi_extremely_sparse            2


## Part 9: Descriptive statistics and optional diagnostics


In [10]:
# Part 9: Descriptive statistics
# Note: to reduce file count, this step does not export diagnostic figures; the statistics table is a tidy CSV containing all requested summary blocks.
stats_rows = []

def append_stat(section: str, **kwargs):
    row = {"section": section}
    row.update(kwargs)
    stats_rows.append(row)

# Mean, median, minimum, and maximum quality_score by region.
for region, g in quality_base.groupby("region", dropna=False):
    append_stat(
        "region_quality_score",
        region=region,
        count=len(g),
        quality_score_mean=g["quality_score"].mean(),
        quality_score_median=g["quality_score"].median(),
        quality_score_min=g["quality_score"].min(),
        quality_score_max=g["quality_score"].max(),
    )

# quality_tier counts by sample_group.
for (sample_group, tier), count in quality_base.groupby(["sample_group", "quality_tier"]).size().items():
    append_stat("sample_group_tier_count", sample_group=sample_group, quality_tier=tier, count=int(count))

# Mean subscore values.
for col in SCORE_WEIGHTS:
    append_stat("component_mean", component=col, mean_score=quality_base[col].mean(), min_score=quality_base[col].min(), max_score=quality_base[col].max())

# Counts for core/sensitivity/excluded_candidate.
for tier, count in quality_base["quality_tier"].value_counts().items():
    append_stat("tier_count", quality_tier=tier, count=int(count))

# China pressure-test city summary.
china = quality_base[quality_base["sample_group"].eq("china_pressure_test")]
append_stat(
    "china_pressure_test_summary",
    sample_group="china_pressure_test",
    count=len(china),
    quality_score_mean=china["quality_score"].mean(),
    quality_score_median=china["quality_score"].median(),
    quality_score_min=china["quality_score"].min(),
    quality_score_max=china["quality_score"].max(),
    core_count=int(china["quality_tier"].eq("core").sum()),
    sensitivity_count=int(china["quality_tier"].eq("sensitivity").sum()),
    excluded_candidate_count=int(china["quality_tier"].eq("excluded_candidate").sum()),
)

stats_table = pd.DataFrame(stats_rows)
print(stats_table.head(12).to_string(index=False))


                section              region  count  quality_score_mean  quality_score_median  quality_score_min  quality_score_max        sample_group quality_tier component  mean_score  min_score  max_score  core_count  sensitivity_count  excluded_candidate_count
   region_quality_score              Africa   12.0           64.745231             64.556460          59.249647          70.101692                 NaN          NaN       NaN         NaN        NaN        NaN         NaN                NaN                       NaN
   region_quality_score        Central Asia    1.0           63.950768             63.950768          63.950768          63.950768                 NaN          NaN       NaN         NaN        NaN        NaN         NaN                NaN                       NaN
   region_quality_score China pressure test    6.0           60.388702             63.020114          51.554208          65.776126                 NaN          NaN       NaN         NaN        NaN        N

## Write Step 7 outputs


In [11]:
# Write Step 7 outputs
# Note: CSV is always written; parquet is written when pyarrow is available. This section does not write to any Step 01-06 or Step 08-13 directories.
def write_table(df: pd.DataFrame, csv_path: Path, parquet_path: Path | None = None) -> None:
    df.to_csv(csv_path, index=False, encoding="utf-8-sig")
    if parquet_path is not None and PARQUET_AVAILABLE:
        df.to_parquet(parquet_path, index=False, engine=PARQUET_ENGINE)

city_quality_cols = [
    "city_id",
    "city_name_en",
    "iso3",
    "region",
    "sample_group",
    "network_integrity_score",
    "historical_maturity_score",
    "poi_completeness_score",
    "population_built_support_score",
    "local_coverage_score",
    "quality_score",
    "quality_tier",
    "quality_weight",
    "quality_confidence_component",
    "review_flag",
    "review_reason",
]
city_quality = quality_base[city_quality_cols].copy()

write_table(quality_base, OUTPUT_PATHS["quality_base_csv"], OUTPUT_PATHS["quality_base_parquet"])
write_table(city_quality, OUTPUT_PATHS["city_quality_csv"], OUTPUT_PATHS["city_quality_parquet"])
write_table(local_quality, OUTPUT_PATHS["local_quality_csv"], OUTPUT_PATHS["local_quality_parquet"])
review_list.to_csv(OUTPUT_PATHS["review_csv"], index=False, encoding="utf-8-sig")
stats_table.to_csv(OUTPUT_PATHS["stats_csv"], index=False, encoding="utf-8-sig")

output_row_counts = {
    "quality_base": len(quality_base),
    "city_quality": len(city_quality),
    "local_quality": len(local_quality),
    "review_list": len(review_list),
    "stats_table": len(stats_table),
}
execution_log["output_row_counts"] = output_row_counts
execution_log["quality_tier_distribution"] = city_quality["quality_tier"].value_counts().to_dict()
execution_log["quality_score_range"] = {
    "min": float(city_quality["quality_score"].min()),
    "max": float(city_quality["quality_score"].max()),
}
execution_log["local_quality_flag_distribution"] = local_quality["local_quality_flag"].value_counts().to_dict()
execution_log["local_use_counts"] = {
    "use_in_local_signature": int(local_quality["use_in_local_signature"].sum()),
    "use_in_local_clustering": int(local_quality["use_in_local_clustering"].sum()),
}
execution_log["review_issue_type_distribution"] = review_list["issue_type"].value_counts().to_dict() if len(review_list) else {}
print(json.dumps(output_row_counts, ensure_ascii=False, indent=2))


{
  "quality_base": 86,
  "city_quality": 86,
  "local_quality": 396670,
  "review_list": 103,
  "stats_table": 24
}


## Part 10: Final self-checks


In [12]:
# Part 10: Final self-checks
# Note: all structural completion criteria are asserted here; the run log is written after assertions pass.
self_checks = {}

def check(name: str, condition: bool, detail: str = ""):
    self_checks[name] = {"passed": bool(condition), "detail": detail}
    assert condition, f"Self-check failed: {name} {detail}"

check("city_quality_scores has 86 rows", len(city_quality) == 86, f"actual={len(city_quality)}")
check("city_id has no duplicates", city_quality["city_id"].is_unique)
check("quality_score has no missing values", city_quality["quality_score"].notna().all())
check("quality_score range is 0-100", city_quality["quality_score"].between(0, 100).all(), f"range=({city_quality['quality_score'].min()}, {city_quality['quality_score'].max()})")
check("quality_confidence_component range is 0-1", city_quality["quality_confidence_component"].between(0, 1).all())
check("quality_tier values are valid", set(city_quality["quality_tier"].unique()).issubset({"core", "sensitivity", "excluded_candidate"}))
check("quality_weight range is 0.25-1.00", city_quality["quality_weight"].between(0.25, 1.00).all())
check("each city has at least one quality subscore", quality_base[list(SCORE_WEIGHTS.keys())].notna().any(axis=1).all())
expected_china = set(city_master.loc[city_master["sample_group"].eq("china_pressure_test"), "city_id"])
actual_china = set(city_quality.loc[city_quality["sample_group"].eq("china_pressure_test"), "city_id"])
check("all China pressure-test cities are retained", expected_china == actual_china, f"expected={sorted(expected_china)}, actual={sorted(actual_china)}")
check("all quality_review_list city_id values exist", set(review_list["city_id"]).issubset(set(city_master["city_id"])))
check("local_quality_flags cover all Step 6 local_morphology_metrics rows", len(local_quality) == len(local_metrics), f"local_quality={len(local_quality)}, local_metrics={len(local_metrics)}")
check("local_quality_flag has no large-scale missingness", local_quality["local_quality_flag"].notna().mean() > 0.999)
for key in ["quality_base_csv", "city_quality_csv", "local_quality_csv", "review_csv", "stats_csv"]:
    check(f"output exists {key}", OUTPUT_PATHS[key].exists(), str(OUTPUT_PATHS[key]))
if PARQUET_AVAILABLE:
    for key in ["quality_base_parquet", "city_quality_parquet", "local_quality_parquet"]:
        check(f"parquet output exists {key}", OUTPUT_PATHS[key].exists(), str(OUTPUT_PATHS[key]))

execution_log["self_checks"] = self_checks
execution_log["finished_at"] = datetime.now().isoformat(timespec="seconds")
execution_log["duration_seconds"] = (datetime.fromisoformat(execution_log["finished_at"]) - datetime.fromisoformat(execution_log["started_at"])).total_seconds()
execution_log["completion_statement"] = "PASS: Step 7 outputs were generated; Step 08/09/10, clustering, morphotype assignment, and final type_confidence were not executed."
with open(OUTPUT_PATHS["execution_log_json"], "w", encoding="utf-8") as f:
    json.dump(execution_log, f, ensure_ascii=False, indent=2)
check("run log exists", OUTPUT_PATHS["execution_log_json"].exists(), str(OUTPUT_PATHS["execution_log_json"]))
# Second write: synchronize the final "run log exists" self-check into the log file.
execution_log["self_checks"] = self_checks
with open(OUTPUT_PATHS["execution_log_json"], "w", encoding="utf-8") as f:
    json.dump(execution_log, f, ensure_ascii=False, indent=2)

print("SELF CHECK PASS")
print("tier distribution:")
print(city_quality["quality_tier"].value_counts().to_string())
print("quality_score range:", float(city_quality["quality_score"].min()), float(city_quality["quality_score"].max()))
print("local flag distribution:")
print(local_quality["local_quality_flag"].value_counts().to_string())
print("review rows:", len(review_list))


SELF CHECK PASS
tier distribution:
quality_tier
core                  48
sensitivity           37
excluded_candidate     1
quality_score range: 51.55420804605241 85.9881863775992
local flag distribution:
local_quality_flag
valid                336479
sparse                35857
boundary_artifact     20481
invalid_metric         3853
review rows: 103
